# 1. Offline Inference

## vLLM Offline Inference 参数分类

### 参数流程
```
LLM.__init__() 参数
        ↓
    EngineArgs (arg_utils.py)
        ↓
    create_engine_config()
        ↓
    VllmConfig (包含多个Config对象)
```

---

### 1. **模型参数 (ModelConfig)**
**文件**: `vllm/config/model.py`

**常用参数**:
- `model`: 模型名称或路径，如 "Qwen/Qwen2.5-7B-Instruct"
- `dtype`: 数据类型，"auto"/"float16"/"bfloat16"/"float32"
- `quantization`: 量化方法，None/"awq"/"gptq"/"fp8"
- `tokenizer`: Tokenizer路径（默认使用model）
- `tokenizer_mode`: "auto"/"hf"/"slow"/"mistral"
- `trust_remote_code`: 是否信任远程代码（bool）
- `max_model_len`: 最大序列长度（自动推断）
- `seed`: 随机种子
- `revision`: 模型版本（分支/标签/commit）
- `skip_tokenizer_init`: 跳过tokenizer初始化

---

### 2. **内存/缓存参数 (CacheConfig)**
**文件**: `vllm/config/cache.py`

**常用参数**:
- `gpu_memory_utilization`: GPU内存使用率，默认0.9（0-1之间）
- `swap_space`: CPU交换空间大小，默认4 GiB
- `kv_cache_memory_bytes`: 手动指定KV cache大小（字节）
- `cache_dtype`: KV cache数据类型，"auto"/"fp8"/"bfloat16"
- `block_size`: 缓存块大小（token数），默认16
- `enable_prefix_caching`: 是否启用前缀缓存，默认True
- `cpu_offload_gb`: CPU offload空间（GiB）
- `kv_offloading_size`: KV cache offload大小（GiB）
- `kv_offloading_backend`: "native"/"lmcache"

---

### 3. **调度参数 (SchedulerConfig)**
**文件**: `vllm/config/scheduler.py`

**常用参数**:
- `max_num_batched_tokens`: 单次迭代最大token数，默认2048
- `max_num_seqs`: 单次迭代最大序列数，默认128
- `enable_chunked_prefill`: 启用分块预填充，默认True
- `max_num_partial_prefills`: 最大部分预填充序列数
- `policy`: 调度策略，"fcfs"（先来先服务）/"priority"
- `async_scheduling`: 异步调度
- `stream_interval`: 流式输出间隔（token数）

---

### 4. **并行参数 (ParallelConfig)**
**文件**: `vllm/config/parallel.py`

**常用参数**:
- `tensor_parallel_size`: 张量并行度（TP），默认1
- `pipeline_parallel_size`: 流水线并行度（PP），默认1
- `data_parallel_size`: 数据并行度（DP），默认1
- `prefill_context_parallel_size`: 预填充上下文并行度
- `disable_custom_all_reduce`: 禁用自定义all-reduce
- `distributed_executor_backend`: 分布式后端，"ray"/"mp"/"uni"
- `enable_expert_parallel`: 启用专家并行（MoE）
- `max_parallel_loading_workers`: 模型加载并行worker数

---

### 5. **加载参数 (LoadConfig)**
**文件**: `vllm/config/load.py`

**常用参数**:
- `load_format`: 加载格式，"auto"/"pt"/"safetensors"/"gguf"/"tensorizer"
- `download_dir`: 模型下载目录
- `safetensors_load_strategy`: "lazy"（内存映射）/"eager"（预加载）
- `model_loader_extra_config`: 加载器额外配置（dict）
- `ignore_patterns`: 忽略的文件模式

---

### 6. **LoRA参数 (LoRAConfig)**
**文件**: `vllm/config/lora.py`

**常用参数**:
- `max_lora_rank`: 最大LoRA秩，默认16
- `max_loras`: 单batch最大LoRA数，默认1
- `max_cpu_loras`: CPU中最大LoRA数
- `lora_dtype`: LoRA数据类型，"auto"/"float16"/"bfloat16"
- `fully_sharded_loras`: 完全分片LoRA

**使用方式**:
```python
from vllm.lora.request import LoRARequest
lora_request = LoRARequest("adapter", 1, "/path/to/lora")
outputs = llm.generate(prompts, lora_request=lora_request)
```

---

### 7. **推测解码 (SpeculativeConfig)**
**文件**: `vllm/config/speculative.py`

**常用参数**:
- `speculative_model`: 推测模型路径
- `num_speculative_tokens`: 推测token数量
- `speculative_draft_tensor_parallel_size`: 推测模型TP大小

**使用方式**:
```python
llm = LLM(
    model="large-model",
    speculative_model="small-draft-model",
    num_speculative_tokens=5
)
```

---

### 8. **结构化输出 (StructuredOutputsConfig)**
**文件**: `vllm/config/structured_outputs.py`

**常用参数**:
- `backend`: 结构化输出后端，默认"auto"
- `disable_fallback`: 禁用回退

**使用方式**:
```python
sampling_params = SamplingParams(
    guided_json={"type": "object", "properties": {...}}
)
```

---

### 9. **编译优化 (CompilationConfig)**
**文件**: `vllm/config/compilation.py`

**常用参数**:
- `level`: 编译级别（0-3）
- `mode`: 编译模式，"vllm_compile"（默认）
- `backend`: 编译后端，"inductor"
- `enable_fusion`: 启用融合优化
- `cudagraph_capture_sizes`: CUDA Graph捕获大小

---

### 10. **采样参数 (SamplingParams)**
**说明**: 这不是Config的一部分，在`generate()`时使用

**常用参数**:
- `temperature`: 温度参数，控制随机性
- `top_p`: nucleus sampling参数
- `top_k`: top-k sampling参数
- `max_tokens`: 最大生成token数
- `presence_penalty`: 存在惩罚
- `frequency_penalty`: 频率惩罚
- `stop`: 停止词列表
- `n`: 每个prompt生成的输出数量
- `logprobs`: 返回log概率数量
- `guided_json`: JSON格式引导
- `guided_regex`: 正则表达式引导
- `guided_choice`: 选择引导

**参考**: https://docs.vllm.ai/en/latest/api/vllm/#vllm.SamplingParams


In [2]:
!nvidia-smi

Wed Nov 26 07:36:16 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:07:00.0 Off |                    0 |
| N/A   30C    P0             54W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
from vllm import LLM, SamplingParams
llm = LLM(model="Qwen/Qwen2.5-7B-Instruct")

/localhome/local-ziruil/vllm-note/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 11-26 07:36:26 [utils.py:253] non-default args: {'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
INFO 11-26 07:36:27 [model.py:631] Resolved architecture: Qwen2ForCausalLM
INFO 11-26 07:36:27 [model.py:1745] Using max model len 32768


2025-11-26 07:36:27,268	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 11-26 07:36:27 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=1365701) INFO 11-26 07:36:28 [core.py:93] Initializing a V1 LLM engine (v0.11.2) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None,

(EngineCore_DP0 pid=1365701) [2025-11-26 07:36:32] INFO _optional_torch_c_dlpack.py:88: JIT-compiling torch-c-dlpack-ext to cache...
(EngineCore_DP0 pid=1365701) /localhome/local-ziruil/vllm-note/venv/lib/python3.12/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:129: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=1365701) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=1365701)   warnings.warn(


(EngineCore_DP0 pid=1365701) INFO 11-26 07:36:35 [cuda.py:418] Valid backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION']
(EngineCore_DP0 pid=1365701) INFO 11-26 07:36:35 [cuda.py:427] Using FLASH_ATTN backend.


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:00<00:01,  1.62it/s]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:01<00:01,  1.32it/s]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:02<00:00,  1.23it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:03<00:00,  1.21it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:03<00:00,  1.25it/s]
(EngineCore_DP0 pid=1365701) 


(EngineCore_DP0 pid=1365701) INFO 11-26 07:36:38 [default_loader.py:314] Loading weights took 3.39 seconds
(EngineCore_DP0 pid=1365701) INFO 11-26 07:36:39 [gpu_model_runner.py:3338] Model loading took 14.2488 GiB memory and 6.119064 seconds
(EngineCore_DP0 pid=1365701) INFO 11-26 07:36:45 [backends.py:631] Using cache directory: /localhome/local-ziruil/.cache/vllm/torch_compile_cache/f1b90306eb/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=1365701) INFO 11-26 07:36:45 [backends.py:647] Dynamo bytecode transform time: 5.66 s
(EngineCore_DP0 pid=1365701) INFO 11-26 07:36:47 [backends.py:210] Directly load the compiled graph(s) for dynamic shape from the cache, took 2.057 s
(EngineCore_DP0 pid=1365701) INFO 11-26 07:36:52 [monitor.py:34] torch.compile takes 7.72 s in total
(EngineCore_DP0 pid=1365701) INFO 11-26 07:36:53 [gpu_worker.py:359] Available KV cache memory: 19.81 GiB
(EngineCore_DP0 pid=1365701) INFO 11-26 07:36:53 [kv_cache_utils.py:1229] GPU KV cache size: 37

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 26.46it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 32.07it/s]


(EngineCore_DP0 pid=1365701) INFO 11-26 07:36:57 [gpu_model_runner.py:4244] Graph capturing finished in 4 secs, took 0.54 GiB
(EngineCore_DP0 pid=1365701) INFO 11-26 07:36:57 [core.py:250] init engine (profile, create kv cache, warmup model) took 17.78 seconds
INFO 11-26 07:36:58 [llm.py:352] Supported tasks: ['generate']


In [4]:
prompts = [
    "Hello, my name is", "Hello, my name is",
    "The purpose of life is to be happy",
    "The future of the world is"
]
#ref:https://docs.vllm.ai/en/latest/api/vllm/#vllm.SamplingParams
sampling_params = SamplingParams(temperature=0.7)
#ref:https://docs.vllm.ai/en/latest/api/vllm/#vllm.LLM.generate
outputs = llm.generate(prompts, sampling_params)
for out in outputs:
    prompt = out.prompt
    output = out.outputs[0].text
    print(f"Prompt: {prompt}")
    print(f"Output: {output}")
    print("\n")


Processed prompts: 100%|██████████| 4/4 [00:00<00:00, 18.20it/s, est. speed input: 109.49 toks/s, output: 291.96 toks/s]

Prompt: Hello, my name is
Output:  Kellie St. Clair. I am a registered psychologist and a member of


Prompt: Hello, my name is
Output:  Brian, and I am the Director of the Adult Sports program at the Fife


Prompt: The purpose of life is to be happy
Output: . But happiness is a poor goal, a shallow goal. It reflects a shallow


Prompt: The future of the world is
Output:  inextricably tied to the health of the oceans. The oceans are a




In [5]:
del llm
llm = LLM(
    model="Qwen/Qwen2.5-7B-Instruct",
    dtype="bfloat16",

    gpu_memory_utilization=0.85,
    swap_space=4,
    enable_prefix_caching=True,
    
    tensor_parallel_size=2,
    
    max_num_batched_tokens=8192,
    max_num_seqs=256,
)

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=512
)

# 执行推理
outputs = llm.generate(prompts, sampling_params)

INFO 11-26 07:36:59 [utils.py:253] non-default args: {'dtype': 'bfloat16', 'tensor_parallel_size': 2, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.85, 'max_num_batched_tokens': 8192, 'max_num_seqs': 256, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
INFO 11-26 07:37:00 [model.py:631] Resolved architecture: Qwen2ForCausalLM
INFO 11-26 07:37:00 [model.py:1745] Using max model len 32768
INFO 11-26 07:37:00 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=1366933) INFO 11-26 07:37:00 [core.py:93] Initializing a V1 LLM engine (v0.11.2) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=2, pipeline_parallel_size=1, data_parallel_size=1, disable

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


(EngineCore_DP0 pid=1366933) (EngineCore_DP0 pid=1366933) INFO 11-26 07:37:04 [parallel_state.py:1208] world_size=2 rank=1 local_rank=1 distributed_init_method=tcp://127.0.0.1:55681 backend=nccl
INFO 11-26 07:37:04 [parallel_state.py:1208] world_size=2 rank=0 local_rank=0 distributed_init_method=tcp://127.0.0.1:55681 backend=nccl
(EngineCore_DP0 pid=1366933) INFO 11-26 07:37:05 [pynccl.py:111] vLLM is using nccl==2.27.5
[Gloo] Rank 0 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1[Gloo] Rank 
1 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1
[Gloo] Rank [Gloo] Rank 01 is connected to  is connected to 11 peer ranks.  peer ranks. Expected number of connected peer ranks is : Expected number of connected peer ranks is : 11

(EngineCore_DP0 pid=1366933) (EngineCore_DP0 pid=1366933) WARNING 11-26 07:37:05 [symm_mem.py:67] SymmMemCommunicator: Device capability 8.0 not supported, communicator is not available.
WARNING 11-26 07:37:05 [s

(EngineCore_DP0 pid=1366933) (Worker_TP1 pid=1366949) [2025-11-26 07:37:07] INFO _optional_torch_c_dlpack.py:88: JIT-compiling torch-c-dlpack-ext to cache...
(EngineCore_DP0 pid=1366933) (Worker_TP0 pid=1366947) [2025-11-26 07:37:07] INFO _optional_torch_c_dlpack.py:88: JIT-compiling torch-c-dlpack-ext to cache...
(EngineCore_DP0 pid=1366933) (Worker_TP1 pid=1366949) /localhome/local-ziruil/vllm-note/venv/lib/python3.12/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:129: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=1366933) (Worker_TP1 pid=1366949) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=1366933) (Worker_TP1 pid=1366949)   warnings.warn(
(EngineCore_DP0 pid=1366933) (Worker_TP0 pid=1366947) /localhome/local-ziruil/vllm-note/venv/lib/python3.12/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:129: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will no

(EngineCore_DP0 pid=1366933) (Worker_TP1 pid=1366949) INFO 11-26 07:37:09 [cuda.py:418] Valid backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION']
(EngineCore_DP0 pid=1366933) (Worker_TP1 pid=1366949) INFO 11-26 07:37:09 [cuda.py:427] Using FLASH_ATTN backend.
(EngineCore_DP0 pid=1366933) (Worker_TP0 pid=1366947) INFO 11-26 07:37:09 [cuda.py:418] Valid backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION']
(EngineCore_DP0 pid=1366933) (Worker_TP0 pid=1366947) INFO 11-26 07:37:09 [cuda.py:427] Using FLASH_ATTN backend.


...

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 28.54it/s]
Capturing CUDA graphs (decode, FULL):  94%|█████████▍| 33/35 [00:01<00:00, 30.73it/s]

(EngineCore_DP0 pid=1366933) (Worker_TP1 pid=1366949) INFO 11-26 07:37:30 [custom_all_reduce.py:216] Registering 4902 cuda graph addresses


Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 29.95it/s]


(EngineCore_DP0 pid=1366933) (Worker_TP0 pid=1366947) INFO 11-26 07:37:30 [custom_all_reduce.py:216] Registering 4902 cuda graph addresses
(EngineCore_DP0 pid=1366933) (Worker_TP0 pid=1366947) INFO 11-26 07:37:31 [gpu_model_runner.py:4244] Graph capturing finished in 4 secs, took 0.53 GiB
(EngineCore_DP0 pid=1366933) INFO 11-26 07:37:31 [core.py:250] init engine (profile, create kv cache, warmup model) took 17.30 seconds
INFO 11-26 07:37:33 [llm.py:352] Supported tasks: ['generate']


Processed prompts: 100%|██████████| 4/4 [00:04<00:00,  1.16s/it, est. speed input: 5.16 toks/s, output: 423.06 toks/s]


In [6]:
from vllm.sampling_params import BeamSearchParams

params = BeamSearchParams(beam_width=5, max_tokens=50)
outputs = llm.beam_search([{"prompt": "Hello, my name is "}], params)

for output in outputs:
    generated_text = output.sequences[0].text
    print(f"Generated text: {generated_text!r}")

Generated text: "Hello, my name is  and I am  years old. I am interested in learning more about . Can you provide me with some information and resources to get started?\nSure, I'd be happy to help! Can you please fill in the blanks with your name and age,"


In [7]:
#clean up environment
del llm

# 2. Generation Model vs. Pooling Model

Generation Model:  文本 → 文本
Pooling Model:     文本 → 向量
```


## vLLM 支持的 Pooling 方法 (runner = "pooling")

Task	APIs
embed	LLM.embed(...), LLM.score(...)*, LLM.encode(..., pooling_task="embed") --语义搜索、文本聚类、推荐系统
classify	LLM.classify(...), LLM.encode(..., pooling_task="classify") --情感分析、主题分类、内容审核
score	LLM.score(...) --搜索重排序、问答匹配、质量评估
token_classify	LLM.reward(...), LLM.encode(..., pooling_task="token_classify") --NER、词性标注、关键词提取
token_embed	LLM.encode(..., pooling_task="token_embed") --词义分析、词向量可视化
plugin	LLM.encode(..., pooling_task="plugin") --自定义任务、扩展功能




## 常见 Embedding 模型

### 英文模型
```python
"sentence-transformers/all-MiniLM-L6-v2"   # 小而快
"sentence-transformers/all-mpnet-base-v2"  # 质量好
"BAAI/bge-large-en-v1.5"                   # SOTA
"intfloat/e5-large-v2"                     # 效果好
```

### 中文/多语言模型
```python
"BAAI/bge-base-zh-v1.5"                    # 中文最佳
"intfloat/multilingual-e5-large"           # 多语言
```




In [8]:
from vllm import LLM

llm = LLM(model="intfloat/e5-small", runner="pooling")
(output,) = llm.embed("Hello, my name is")

embeds = output.outputs.embedding
print(f"Embeddings: {embeds!r} (size={len(embeds)})")
del llm

INFO 11-26 07:37:44 [utils.py:253] non-default args: {'runner': 'pooling', 'disable_log_stats': True, 'model': 'intfloat/e5-small'}
INFO 11-26 07:37:44 [config.py:896] Found sentence-transformers tokenize configuration.
INFO 11-26 07:37:44 [model.py:631] Resolved architecture: BertModel
INFO 11-26 07:37:44 [config.py:784] Found sentence-transformers modules configuration.
INFO 11-26 07:37:44 [config.py:815] Found pooling configuration.
INFO 11-26 07:37:44 [model.py:1968] Downcasting torch.float32 to torch.float16.
INFO 11-26 07:37:45 [model.py:1745] Using max model len 512
WARNING 11-26 07:37:45 [vllm.py:486] Pooling models do not support full cudagraphs. Overriding cudagraph_mode to PIECEWISE.
INFO 11-26 07:37:45 [vllm.py:582] Only "last" pooling supports chunked prefill and prefix caching; disabling both.
(EngineCore_DP0 pid=1367952) INFO 11-26 07:37:45 [core.py:93] Initializing a V1 LLM engine (v0.11.2) with config: model='intfloat/e5-small', speculative_config=None, tokenizer='intf

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


(EngineCore_DP0 pid=1367952) INFO 11-26 07:37:48 [parallel_state.py:1208] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://10.176.187.103:47553 backend=nccl
(EngineCore_DP0 pid=1367952) INFO 11-26 07:37:49 [parallel_state.py:1394] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=1367952) INFO 11-26 07:37:49 [gpu_model_runner.py:3259] Starting to load model intfloat/e5-small...


(EngineCore_DP0 pid=1367952) [2025-11-26 07:37:50] INFO _optional_torch_c_dlpack.py:88: JIT-compiling torch-c-dlpack-ext to cache...
(EngineCore_DP0 pid=1367952) /localhome/local-ziruil/vllm-note/venv/lib/python3.12/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:129: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=1367952) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=1367952)   warnings.warn(


(EngineCore_DP0 pid=1367952) INFO 11-26 07:37:52 [cuda.py:418] Valid backends: ['FLASH_ATTN', 'FLEX_ATTENTION']
(EngineCore_DP0 pid=1367952) INFO 11-26 07:37:52 [cuda.py:427] Using FLASH_ATTN backend.
(EngineCore_DP0 pid=1367952) INFO 11-26 07:37:52 [weight_utils.py:481] No model.safetensors.index.json found in remote.


...

(EngineCore_DP0 pid=1367952) INFO 11-26 07:37:55 [backends.py:631] Using cache directory: /localhome/local-ziruil/.cache/vllm/torch_compile_cache/a88f60dd63/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=1367952) INFO 11-26 07:37:55 [backends.py:647] Dynamo bytecode transform time: 1.76 s
(EngineCore_DP0 pid=1367952) INFO 11-26 07:37:56 [backends.py:210] Directly load the compiled graph(s) for dynamic shape from the cache, took 0.356 s
(EngineCore_DP0 pid=1367952) INFO 11-26 07:37:56 [monitor.py:34] torch.compile takes 2.12 s in total


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 12.97it/s]


(EngineCore_DP0 pid=1367952) INFO 11-26 07:37:58 [gpu_model_runner.py:4244] Graph capturing finished in 5 secs, took 0.16 GiB
(EngineCore_DP0 pid=1367952) INFO 11-26 07:37:58 [core.py:250] init engine (profile, create kv cache, warmup model) took 4.71 seconds
(EngineCore_DP0 pid=1367952) INFO 11-26 07:37:58 [core.py:125] Disabling chunked prefill for model without KVCache
(EngineCore_DP0 pid=1367952) INFO 11-26 07:37:58 [vllm.py:582] Only "last" pooling supports chunked prefill and prefix caching; disabling both.
INFO 11-26 07:37:58 [llm.py:352] Supported tasks: ['embed', 'token_embed']


Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 181.41it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Embeddings: [-0.007237747777253389, -0.0022512588184326887, -0.1356087327003479, -0.008090504445135593, 0.00413373252376914, -0.07522836327552795, -0.019572092220187187, 0.026297971606254578, -0.03687022626399994, 0.047211870551109314, -0.0440126396715641, 0.006497856229543686, 0.06343468278646469, -0.03445396572351456, 0.001397340907715261, -0.030711311846971512, 0.034255173057317734, 0.03800990432500839, -0.030342526733875275, 0.02024971693754196, -0.1126604750752449, -0.007830869406461716, 0.025162821635603905, 0.05089786276221275, 0.04455699399113655, 0.032827410846948624, -0.0008300268091261387, 0.06656517088413239, 0.02381030097603798, -0.009227977134287357, -0.027510223910212517, -0.052506763488054276, -0.0387493334710598, 0.0667518898844719, -0.028793074190616608, 0.0051481230184435844, -0.04944176599383354, 0.042035650461912155, 0.024207882583141327, 0.051341891288757324, -0.0946764126420021, -0.009023148566484451, 0.033353183418512344, 0.06984800100326538, -0.0199552457779645

In [9]:
from vllm import LLM

llm = LLM(model="BAAI/bge-reranker-v2-m3", runner="pooling")
(output,) = llm.score(
    "What is the capital of France?",
    "The capital of Brazil is Brasilia.",
)

score = output.outputs.score
print(f"Score: {score}")
del llm

INFO 11-26 07:37:59 [utils.py:253] non-default args: {'runner': 'pooling', 'disable_log_stats': True, 'model': 'BAAI/bge-reranker-v2-m3'}
INFO 11-26 07:38:00 [model.py:631] Resolved architecture: XLMRobertaForSequenceClassification
INFO 11-26 07:38:00 [model.py:1968] Downcasting torch.float32 to torch.float16.
INFO 11-26 07:38:00 [model.py:1745] Using max model len 8192
INFO 11-26 07:38:00 [vllm.py:582] Only "last" pooling supports chunked prefill and prefix caching; disabling both.
(EngineCore_DP0 pid=1368566) INFO 11-26 07:38:01 [core.py:93] Initializing a V1 LLM engine (v0.11.2) with config: model='BAAI/bge-reranker-v2-m3', speculative_config=None, tokenizer='BAAI/bge-reranker-v2-m3', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantizat

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


(EngineCore_DP0 pid=1368566) INFO 11-26 07:38:05 [parallel_state.py:1208] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://10.176.187.103:36023 backend=nccl
(EngineCore_DP0 pid=1368566) INFO 11-26 07:38:05 [parallel_state.py:1394] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=1368566) INFO 11-26 07:38:05 [gpu_model_runner.py:3259] Starting to load model BAAI/bge-reranker-v2-m3...


(EngineCore_DP0 pid=1368566) [2025-11-26 07:38:06] INFO _optional_torch_c_dlpack.py:88: JIT-compiling torch-c-dlpack-ext to cache...
(EngineCore_DP0 pid=1368566) /localhome/local-ziruil/vllm-note/venv/lib/python3.12/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:129: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=1368566) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=1368566)   warnings.warn(


(EngineCore_DP0 pid=1368566) INFO 11-26 07:38:08 [cuda.py:418] Valid backends: ['FLASH_ATTN', 'FLEX_ATTENTION']
(EngineCore_DP0 pid=1368566) INFO 11-26 07:38:08 [cuda.py:427] Using FLASH_ATTN backend.
(EngineCore_DP0 pid=1368566) INFO 11-26 07:38:08 [weight_utils.py:481] No model.safetensors.index.json found in remote.


...

(EngineCore_DP0 pid=1368566) INFO 11-26 07:38:12 [backends.py:631] Using cache directory: /localhome/local-ziruil/.cache/vllm/torch_compile_cache/8152efcac9/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=1368566) INFO 11-26 07:38:12 [backends.py:647] Dynamo bytecode transform time: 2.80 s
(EngineCore_DP0 pid=1368566) INFO 11-26 07:38:13 [backends.py:210] Directly load the compiled graph(s) for dynamic shape from the cache, took 0.663 s
(EngineCore_DP0 pid=1368566) INFO 11-26 07:38:15 [monitor.py:34] torch.compile takes 3.47 s in total


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:06<00:00,  8.00it/s]


(EngineCore_DP0 pid=1368566) INFO 11-26 07:38:16 [gpu_model_runner.py:4244] Graph capturing finished in 7 secs, took 0.27 GiB
(EngineCore_DP0 pid=1368566) INFO 11-26 07:38:16 [core.py:250] init engine (profile, create kv cache, warmup model) took 7.14 seconds
(EngineCore_DP0 pid=1368566) INFO 11-26 07:38:18 [core.py:125] Disabling chunked prefill for model without KVCache
(EngineCore_DP0 pid=1368566) INFO 11-26 07:38:18 [vllm.py:582] Only "last" pooling supports chunked prefill and prefix caching; disabling both.
INFO 11-26 07:38:18 [llm.py:352] Supported tasks: ['classify', 'token_classify', 'score']


Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 69.04it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Score: 0.001092187943868339


In [14]:
## 完整工作流示例

### 场景：智能客服系统

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

print("=" * 70)
print("智能客服系统 - 两阶段检索（Embedding + Reranking）")
print("=" * 70)

# ========== 第一步：准备知识库 ==========
knowledge_base = [
    "如何重置密码？点击忘记密码链接，然后输入邮箱获取验证码",
    "如何联系客服？拨打400热线或在线客服",
    "退款需要多久？3-5个工作日到账",
    "如何修改个人信息？登录后进入个人中心进行修改",
    "订单怎么取消？在订单详情页点击取消订单",
    "如何查看物流信息？在订单页面点击查看物流",
    "支持哪些支付方式？支付宝、微信、银行卡",
    "如何申请发票？在订单页面申请电子发票",
]

print("\n📚 知识库 ({} 条)".format(len(knowledge_base)))

# ========== 第二步：加载模型 ==========
print("\n🔄 加载 Embedding 模型...")
llm_embed = LLM(
    model="BAAI/bge-base-zh-v1.5",
    task="embed",
    gpu_memory_utilization=0.5,  # 留一些显存给 reranker
)

# ========== 第三步：生成知识库向量 ==========
print("📊 生成知识库向量...")
kb_outputs = llm_embed.embed(knowledge_base)
kb_embeddings = np.array([output.outputs.embedding for output in kb_outputs])
print(f"✅ 向量维度: {kb_embeddings.shape}")

# ========== 第四步：用户查询 ==========
user_query = "忘记密码怎么办"
print(f"\n❓ 用户问题: {user_query}")

# ========== 第五步：粗排（Embedding 相似度）==========
print("\n🔍 阶段1: 粗排（Embedding 相似度）")
query_output = llm_embed.embed([user_query])
query_emb = np.array([query_output[0].outputs.embedding])

similarities = cosine_similarity(query_emb, kb_embeddings)[0]

# 选择 Top-3 候选
top_k = 3
top_k_indices = similarities.argsort()[-top_k:][::-1]
candidates = [knowledge_base[i] for i in top_k_indices]

print(f"   Top-{top_k} 候选：")
for rank, idx in enumerate(top_k_indices, 1):
    print(f"   {rank}. [{similarities[idx]:.4f}] {knowledge_base[idx]}")

# ========== 第六步：精排（Reranker）==========
print(f"\n🎯 阶段2: 精排（Reranker）")

# 加载 Reranker
print("   加载 Reranker 模型...")
llm_rerank = LLM(
    model="BAAI/bge-reranker-base",
    task="score",
    gpu_memory_utilization=0.4,
)

# 对候选答案重排序
queries = [user_query] * len(candidates)
score_outputs = llm_rerank.score(queries, candidates)
scores = np.array([output.outputs.score for output in score_outputs])

print(f"   Reranker 分数：")
for rank, (cand, score) in enumerate(zip(candidates, scores), 1):
    print(f"   {rank}. [{score:.4f}] {cand}")

# ========== 第七步：返回最佳答案 ==========
best_idx = scores.argmax()
best_answer = candidates[best_idx]
best_score = scores[best_idx]

print("\n" + "=" * 70)
print(f"✨ 最终答案: {best_answer}")
print(f"   置信度: {best_score:.4f}")
print("=" * 70)

智能客服系统 - 两阶段检索（Embedding + Reranking）

📚 知识库 (8 条)

🔄 加载 Embedding 模型...
INFO 11-26 08:07:02 [utils.py:253] non-default args: {'task': 'embed', 'gpu_memory_utilization': 0.5, 'disable_log_stats': True, 'model': 'BAAI/bge-base-zh-v1.5'}
INFO 11-26 08:07:03 [model.py:631] Resolved architecture: BertModel
INFO 11-26 08:07:03 [model.py:1968] Downcasting torch.float32 to torch.float16.
INFO 11-26 08:07:03 [model.py:1745] Using max model len 512
INFO 11-26 08:07:03 [arg_utils.py:1443] Using ray runtime env (env vars redacted): {}
INFO 11-26 08:07:03 [vllm.py:582] Only "last" pooling supports chunked prefill and prefix caching; disabling both.
(EngineCore_DP0 pid=1529901) INFO 11-26 08:07:03 [core.py:93] Initializing a V1 LLM engine (v0.11.2) with config: model='BAAI/bge-base-zh-v1.5', speculative_config=None, tokenizer='BAAI/bge-base-zh-v1.5', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=512,

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


(EngineCore_DP0 pid=1529901) INFO 11-26 08:07:04 [parallel_state.py:1208] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://10.176.187.103:35235 backend=nccl
(EngineCore_DP0 pid=1529901) INFO 11-26 08:07:04 [parallel_state.py:1394] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=1529901) INFO 11-26 08:07:05 [gpu_model_runner.py:3259] Starting to load model BAAI/bge-base-zh-v1.5...


(EngineCore_DP0 pid=1529901) [2025-11-26 08:07:06] INFO _optional_torch_c_dlpack.py:88: JIT-compiling torch-c-dlpack-ext to cache...
(EngineCore_DP0 pid=1529901) /localhome/local-ziruil/vllm-note/venv/lib/python3.12/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:129: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=1529901) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=1529901)   warnings.warn(


(EngineCore_DP0 pid=1529901) INFO 11-26 08:07:08 [cuda.py:418] Valid backends: ['FLASH_ATTN', 'FLEX_ATTENTION']
(EngineCore_DP0 pid=1529901) INFO 11-26 08:07:08 [cuda.py:427] Using FLASH_ATTN backend.


...

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:06<00:00,  8.30it/s]


(EngineCore_DP0 pid=1531005) INFO 11-26 08:07:41 [gpu_model_runner.py:4244] Graph capturing finished in 7 secs, took 0.51 GiB
(EngineCore_DP0 pid=1531005) INFO 11-26 08:07:41 [core.py:250] init engine (profile, create kv cache, warmup model) took 7.09 seconds
(EngineCore_DP0 pid=1531005) INFO 11-26 08:07:43 [core.py:125] Disabling chunked prefill for model without KVCache
(EngineCore_DP0 pid=1531005) INFO 11-26 08:07:43 [vllm.py:582] Only "last" pooling supports chunked prefill and prefix caching; disabling both.
INFO 11-26 08:07:44 [llm.py:352] Supported tasks: ['classify', 'token_classify', 'score']


Processed prompts: 100%|██████████| 3/3 [00:00<00:00, 324.78it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

   Reranker 分数：
   1. [0.7461] 如何重置密码？点击忘记密码链接，然后输入邮箱获取验证码
   2. [0.0283] 如何修改个人信息？登录后进入个人中心进行修改
   3. [0.0036] 如何联系客服？拨打400热线或在线客服

✨ 最终答案: 如何重置密码？点击忘记密码链接，然后输入邮箱获取验证码
   置信度: 0.7461


ERROR 11-26 08:10:32 [core_client.py:598] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.
ERROR 11-26 08:10:38 [core_client.py:598] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.


# 3.Ray Data LLM
ref:https://docs.ray.io/en/latest/data/working-with-llms.html
## 优势
    ### 流式处理超过内存的数据集
    ### 自动分片和负载均衡以及自动容错
    ### 持续批处理，GPU利用率最大化
## 使用场景
   一般是多机部署的情况，但单机数据量超大或多个vLLM实例或需要自动容错也可以考虑ray

vllm-note/User_guide/ray_offline_inference.py